# Hurst Exponent

The Hurst exponent H tells you whether a price series is mean-reverting (H < 0.5), random walk (H ≈ 0.5), or trending (H > 0.5). Chan uses the Rescaled Range (R/S) method to calculate it.

**The math:**

For a time series of length N, split it into sub-periods of length n. For each sub-period:

- Compute the mean-adjusted cumulative deviation
- Compute R = range of that cumulative deviation
- Compute S = standard deviation of the sub-period
- Average R/S across all sub-periods of the same n
- Do this for multiple values of n, then fit: log(R/S) = H * log(n) + C
H is the slope of that log-log regression

In [1]:
import numpy as np

In [9]:
def compute_rs(series):
    n = len(series)
    mean = np.mean(series)

    deviations = series - mean
    cummulative = np.cumsum(deviations)
    R = np.max(cummulative) - np.min(cummulative)
    S = np.std(series, ddof=1)
    if S == 0:
        return 0
    return R / S

def hurst_exponent(price_series):
    returns = np.log(price_series[1:] / price_series[:-1])
    N = len(returns)
    chunk_sizes = range(10, N//2, 10)
    rs_values = []
    ns = []

    for n in chunk_sizes:
        num_chunks = N // n
        if num_chunks < 2:
            continue
        
        rs_for_this_n = []
        for i in range(num_chunks):
            chunk = returns[i*n : (i+1)*n]
            rs = compute_rs(chunk)
            if rs > 0:
                rs_for_this_n.append(rs)
        
        if rs_for_this_n:
            rs_values.append(np.mean(rs_for_this_n))
            ns.append(n)
    
    # Step 4: fit log(R/S) = H * log(n) + C
    log_ns = np.log(ns)
    log_rs = np.log(rs_values)
    
    # H is just the slope of this line
    H, _ = np.polyfit(log_ns, log_rs, 1)
    
    return H

In [11]:
np.random.seed(42)
n = 2000  # longer series = more reliable H estimate

# Mean-reverting: use an Ornstein-Uhlenbeck process
# Each step pulls back toward 0 with strength theta
theta = 0.3
ou = np.zeros(n)
for i in range(1, n):
    ou[i] = ou[i-1] - theta * ou[i-1] + np.random.randn()
# Convert to prices (must be positive)
mean_reverting = np.exp(ou * 0.1 + 4.6)  # centered around ~100

# True random walk
random_walk = 100 + np.cumsum(np.random.randn(n))

# Trending: each step has a positive bias
trending = 100 + np.cumsum(np.random.randn(n) + 0.08)

print(f"Mean-reverting H: {hurst_exponent(mean_reverting):.3f}")  # expect < 0.5
print(f"Random walk H:    {hurst_exponent(random_walk):.3f}")     # expect ≈ 0.5
print(f"Trending H:       {hurst_exponent(trending):.3f}")        # expect > 0.5

Mean-reverting H: 0.167
Random walk H:    0.555
Trending H:       0.572


In [14]:
import yfinance as yf
tickers = ["KO","PEP","BTC-USD","SPY"]
data = yf.download(tickers, start="2019-01-01", end="2024-01-01", auto_adjust=True)["Close"]
data = data.dropna()

[*********************100%***********************]  4 of 4 completed


In [22]:
for ticker in ["KO", "BTC-USD", "SPY"]:
    H = hurst_exponent(data[ticker].values)
    if H < 0.45:
        regime = "mean-reverting"
    elif H > 0.55:
        regime = "trending"
    else:
        regime = "random walk"
    print(f"{ticker:<10} H = {H:.3f}  -> {regime}")

KO         H = 0.511  -> random walk
BTC-USD    H = 0.632  -> trending
SPY        H = 0.555  -> trending
